# SF Flattest Route Finder
Finds the flattest cycling route between two points in San Francisco.
Data sourced live from DataSF: street centerlines + elevation contours.

**Sections:**
1. Fetch data from DataSF API
2. Build node/edge tables from street segment endpoints
3. Assign elevation to nodes via contour interpolation
4. Cache to SQLite (skip fetch next time)
5. Route — Dijkstra minimizing elevation gain
6. Visualize on map

In [ ]:
# pip install geopandas pandas numpy networkx folium scipy requests shapely
import requests
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
import folium
from scipy.spatial import cKDTree
import sqlite3
import warnings
warnings.filterwarnings('ignore')

DB_PATH = "sf_routes.db"

## 1. Fetch Data from DataSF
Skip this section and jump to **Section 5** if you've already built the cache.

In [ ]:
def fetch_all(dataset_id, where=None, limit=50000):
    """Paginate through a DataSF Socrata GeoJSON endpoint."""
    url = f"https://data.sfgov.org/resource/{dataset_id}.geojson"
    features = []
    offset = 0
    while True:
        params = {"$limit": limit, "$offset": offset}
        if where:
            params["$where"] = where
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        batch = r.json().get("features", [])
        features.extend(batch)
        print(f"  {dataset_id}: {len(features):,} records")
        if len(batch) < limit:
            break
        offset += limit
    return {"type": "FeatureCollection", "features": features}

In [ ]:
print("Fetching active streets...")
streets = gpd.GeoDataFrame.from_features(
    fetch_all("3psu-pn9h", where="active=true")["features"], crs="EPSG:4326"
)
print(f"  {len(streets):,} active street segments\n")

print("Fetching elevation contours...")
contours = gpd.GeoDataFrame.from_features(
    fetch_all("6d73-6c4f")["features"], crs="EPSG:4326"
)
contours["elevation"] = pd.to_numeric(contours["elevation"], errors="coerce")
print(f"  {len(contours):,} contour lines")
print(f"  Elevation range: {contours['elevation'].min():.0f} to {contours['elevation'].max():.0f} (verify units below)")

## 2. Build Node and Edge Tables
Each street segment has a `f_node_cnn` (from-node) and `t_node_cnn` (to-node).
We use the first and last coordinate of each LineString as the node positions,
and skip intermediate vertices for now.

In [ ]:
nodes = {}  # {node_cnn: (lon, lat)}
edges = []

for _, row in streets.iterrows():
    geom = row.geometry
    if geom is None:
        continue
    f_cnn = row.get("f_node_cnn")
    t_cnn = row.get("t_node_cnn")
    if pd.isna(f_cnn) or pd.isna(t_cnn):
        continue

    if geom.geom_type == "MultiLineString":
        start = list(geom.geoms[0].coords)[0]
        end   = list(geom.geoms[-1].coords)[-1]
    else:
        coords = list(geom.coords)
        start, end = coords[0], coords[-1]

    f, t = str(f_cnn), str(t_cnn)
    nodes[f] = start  # (lon, lat)
    nodes[t] = end
    edges.append((f, t, row.get("streetname", "")))

nodes_df = pd.DataFrame(
    [(nid, lon, lat) for nid, (lon, lat) in nodes.items()],
    columns=["node_id", "lon", "lat"]
)
edges_df = pd.DataFrame(edges, columns=["f_node", "t_node", "streetname"])

print(f"Nodes: {len(nodes_df):,}")
print(f"Edges: {len(edges_df):,}")
nodes_df.head()

## 3. Assign Elevation to Nodes
Sample contour lines at ~20m intervals, build a KD-tree, then use
inverse-distance weighting (IDW) to interpolate elevation at each node.

Everything is projected to UTM Zone 10N (EPSG:26910) so distances are in meters.

In [ ]:
contours_utm = contours.to_crs("EPSG:26910")
nodes_gdf = gpd.GeoDataFrame(
    nodes_df,
    geometry=gpd.points_from_xy(nodes_df.lon, nodes_df.lat),
    crs="EPSG:4326"
).to_crs("EPSG:26910")

print("Sampling contour lines at 20m intervals (may take ~1 min)...")
pts, elevs = [], []
for _, row in contours_utm.iterrows():
    elev = row["elevation"]
    geom = row.geometry
    if geom is None or pd.isna(elev):
        continue
    n = max(2, int(geom.length / 20))
    for i in range(n):
        pt = geom.interpolate(i / (n - 1), normalized=True)
        pts.append([pt.x, pt.y])
        elevs.append(elev)

pts   = np.array(pts)
elevs = np.array(elevs)
print(f"  {len(pts):,} sample points from contours")

# IDW: find 6 nearest contour samples per node, weight by 1/distance
tree = cKDTree(pts)
node_xy = np.column_stack([nodes_gdf.geometry.x, nodes_gdf.geometry.y])
dists, idxs = tree.query(node_xy, k=6)

w = np.where(dists == 0, 1e10, 1.0 / dists)
nodes_df["elevation"] = (w * elevs[idxs]).sum(axis=1) / w.sum(axis=1)

print(f"Node elevation range: {nodes_df['elevation'].min():.1f} to {nodes_df['elevation'].max():.1f}")
print("Note: check units match contour source (DataSF lists elevation in feet for this dataset)")
nodes_df.head()

## 4. Compute Edge Weights and Save to SQLite

In [ ]:
node_elev = nodes_df.set_index("node_id")["elevation"]
node_lat  = nodes_df.set_index("node_id")["lat"]
node_lon  = nodes_df.set_index("node_id")["lon"]

edges_df["elev_f"] = edges_df["f_node"].map(node_elev)
edges_df["elev_t"] = edges_df["t_node"].map(node_elev)

# Elevation gain = only uphill (relevant for cycling effort)
# Elevation change = absolute difference (useful alternative)
edges_df["elev_gain"]   = (edges_df["elev_t"] - edges_df["elev_f"]).clip(lower=0)
edges_df["elev_change"] = (edges_df["elev_t"] - edges_df["elev_f"]).abs()

# Haversine edge distance
lat1 = edges_df["f_node"].map(node_lat).values
lat2 = edges_df["t_node"].map(node_lat).values
lon1 = edges_df["f_node"].map(node_lon).values
lon2 = edges_df["t_node"].map(node_lon).values
R    = 6_371_000
dlat = np.radians(lat2 - lat1)
dlon = np.radians(lon2 - lon1)
a    = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
edges_df["distance_m"] = 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

conn = sqlite3.connect(DB_PATH)
nodes_df.to_sql("nodes", conn, if_exists="replace", index=False)
edges_df.to_sql("edges", conn, if_exists="replace", index=False)
conn.close()
print(f"Saved to {DB_PATH}")
edges_df.head()

---
## 5. Load Graph from Cache
Start here on repeat runs — no need to re-fetch or re-process.

In [ ]:
conn = sqlite3.connect(DB_PATH)
nodes_df = pd.read_sql("SELECT * FROM nodes", conn)
edges_df = pd.read_sql("SELECT * FROM edges", conn)
conn.close()

G = nx.Graph()

for _, r in nodes_df.iterrows():
    G.add_node(r["node_id"], lat=r["lat"], lon=r["lon"], elevation=r["elevation"])

valid_edges = edges_df.dropna(subset=["elev_f", "elev_t"])
for _, r in valid_edges.iterrows():
    G.add_edge(
        r["f_node"], r["t_node"],
        elev_gain=r["elev_gain"],
        elev_change=r["elev_change"],
        distance_m=r["distance_m"],
        streetname=r["streetname"]
    )

n_components = nx.number_connected_components(G)
print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Connected components: {n_components}  (>1 means some streets are isolated — usually fine)")

## 6. Route
Set `ORIGIN` and `DEST` as `(lat, lon)` tuples. The cell snaps each to the nearest graph node,
then runs Dijkstra twice: once minimizing **elevation gain**, once minimizing **distance**.

In [ ]:
# --- Edit these ---
ORIGIN = (37.7509, -122.4337)  # Noe Valley
DEST   = (37.8044, -122.4277)  # Marina District
# ------------------

def nearest_node(G, lat, lon):
    ndata = [(n, d["lat"], d["lon"]) for n, d in G.nodes(data=True)]
    lats  = np.array([d[1] for d in ndata])
    lons  = np.array([d[2] for d in ndata])
    dists = np.hypot(lats - lat, lons - lon)
    return ndata[int(np.argmin(dists))][0]

origin_node = nearest_node(G, *ORIGIN)
dest_node   = nearest_node(G, *DEST)

o = G.nodes[origin_node]
d = G.nodes[dest_node]
print(f"Origin : {origin_node}  ({o['lat']:.4f}, {o['lon']:.4f})  elev {o['elevation']:.1f}")
print(f"Dest   : {dest_node}    ({d['lat']:.4f}, {d['lon']:.4f})  elev {d['elevation']:.1f}")

if not nx.has_path(G, origin_node, dest_node):
    print("\nWARNING: No path found — origin and destination are in different connected components.")
    print("Try adjusting the coordinates slightly.")
else:
    path_flat  = nx.dijkstra_path(G, origin_node, dest_node, weight="elev_gain")
    path_short = nx.dijkstra_path(G, origin_node, dest_node, weight="distance_m")

    def route_stats(G, path, label):
        gain  = sum(G[path[i]][path[i+1]]["elev_gain"]   for i in range(len(path)-1))
        dist  = sum(G[path[i]][path[i+1]]["distance_m"]  for i in range(len(path)-1))
        print(f"{label:10s}  {dist/1000:.2f} km   {gain:.0f} elevation gain   ({len(path)-1} segments)")

    print()
    route_stats(G, path_flat,  "Flattest")
    route_stats(G, path_short, "Shortest")

## 7. Visualize
Blue = flattest route, red = shortest route.

In [ ]:
def path_coords(G, path):
    return [(G.nodes[n]["lat"], G.nodes[n]["lon"]) for n in path]

center = [(ORIGIN[0] + DEST[0]) / 2, (ORIGIN[1] + DEST[1]) / 2]
m = folium.Map(location=center, zoom_start=14, tiles="CartoDB positron")

folium.PolyLine(path_coords(G, path_short), color="#e74c3c", weight=5, opacity=0.75, tooltip="Shortest").add_to(m)
folium.PolyLine(path_coords(G, path_flat),  color="#2980b9", weight=5, opacity=0.75, tooltip="Flattest").add_to(m)

folium.Marker(list(ORIGIN), popup="Origin",      icon=folium.Icon(color="green", icon="play")).add_to(m)
folium.Marker(list(DEST),   popup="Destination", icon=folium.Icon(color="red",   icon="stop")).add_to(m)

legend = """
<div style="position:fixed;bottom:30px;left:30px;background:white;padding:12px 16px;
            border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);font-size:13px;z-index:999">
  <b>Routes</b><br>
  <span style="color:#2980b9">&#9644;&#9644;</span> Flattest (min elevation gain)<br>
  <span style="color:#e74c3c">&#9644;&#9644;</span> Shortest (min distance)
</div>
"""
m.get_root().html.add_child(folium.Element(legend))
m